# Notebook 03 — SOC par Filtre de Kalman Étendu (EKF)

L'EKF est un **observateur d'état non-linéaire** — exactement ce que tu as vu en automatique.

**Système :**
- État : x = [SOC, Vc1, Vc2]
- Entrée : u = I (courant mesuré)
- Mesure : z = V_terminal

**Pourquoi EKF ?** Le Coulomb Counting dérive dans le temps.
L'EKF corrige cette dérive en fusionnant le modèle ECM avec la mesure de tension.


## 0. Imports

In [1]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import json, warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))
DATA_PROC = PROJECT_ROOT / 'data' / 'processed'

# Charger données
df_all  = pd.read_parquet(DATA_PROC / 'nasa_all_cycles.parquet')
df_summ = pd.read_parquet(DATA_PROC / 'nasa_summary.parquet')

# Charger modèle ECM
with open(PROJECT_ROOT / 'ml' / 'models' / 'ecm_b0005.json') as f:
    ecm = json.load(f)

Q_NOM  = ecm['Q_nom_Ah']
R0     = ecm['R0_ohm']
R1     = ecm['R1_ohm']
C1     = ecm['C1_F']
R2     = ecm['R2_ohm']
C2     = ecm['C2_F']
poly   = np.poly1d(ecm['ocv_poly_coeffs'])

print('Parametres ECM charges :')
for k, v in ecm.items():
    if 'poly' not in k:
        print(f'  {k} = {v}')


Parametres ECM charges :
  cell = B0005
  Q_nom_Ah = 2.0
  R0_ohm = 0.001
  R1_ohm = 0.002894460995795434
  C1_F = 1999.9999999999072
  R2_ohm = 0.001
  C2_F = 19999.999999999836
  tau1_s = 5.7889219915906
  tau2_s = 19.999999999999837
  validation_rmse_mV = 69.17359418707962


## 1. Implémentation EKF

Le module `estimation/soc/kalman/ekf.py` est déjà créé.
On l'importe et on l'utilise directement ici.


In [2]:
from estimation.soc.kalman.ekf import BatteryEKF

# Instanciation de l'EKF avec les paramètres ECM identifiés
ekf = BatteryEKF(
    R0=R0, R1=R1, C1=C1, R2=R2, C2=C2,
    Q_nom_Ah=Q_NOM,
    ocv_poly_coeffs=ecm['ocv_poly_coeffs'],
    Q_noise=np.diag([1e-6, 1e-6, 1e-6]),   # bruit de processus
    R_noise=np.array([[1e-3]]),              # bruit de mesure (variance V)
    soc0=1.0
)
print('EKF instancié — état initial x0 =', ekf.x.flatten())


EKF instancié — état initial x0 = [1. 0. 0.]


## 2. Estimation SOC sur un cycle de décharge

In [3]:
disch = df_all[(df_all['cell'] == 'B0005') & (df_all['type'] == 'discharge')]
cids  = disch['cycle_idx'].unique()
cycle = disch[disch['cycle_idx'] == cids[9]].copy()  # cycle #10

t = cycle['time_s'].values.astype(float)
I = cycle['current_A'].values.astype(float)
V = cycle['voltage_V'].values.astype(float)

# Coulomb Counting (référence)
Q_As  = Q_NOM * 3600
dt_arr = np.diff(t, prepend=t[0])
soc_cc = np.clip(1.0 + np.cumsum(I * dt_arr) / Q_As, 0, 1)

# EKF
ekf.reset(soc0=1.0)
soc_ekf = np.zeros(len(t))
P_std   = np.zeros(len(t))

for k in range(len(t)):
    dt = dt_arr[k]
    ekf.predict(I[k], dt)
    ekf.update(V[k])
    soc_ekf[k] = ekf.x[0, 0]
    P_std[k]   = np.sqrt(ekf.P[0, 0])

# Figure
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
    subplot_titles=('SOC estimé — CC vs EKF', 'Incertitude EKF (1-sigma)'),
    vertical_spacing=0.1)

t_min = (t - t[0]) / 60
fig.add_trace(go.Scatter(x=t_min, y=soc_cc*100, name='Coulomb Counting',
    line=dict(color='royalblue', dash='dot', width=1.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=t_min, y=soc_ekf*100, name='EKF',
    line=dict(color='crimson', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=t_min, y=P_std*100, name='Std SOC',
    line=dict(color='seagreen', width=1.5),
    fill='tozeroy', fillcolor='rgba(46,160,87,0.2)'), row=2, col=1)

fig.update_layout(title='SOC EKF vs Coulomb Counting — B0005 Cycle #10',
    xaxis2_title='Temps (min)', template='plotly_white', height=500)
fig.show()
fig.write_html(str(DATA_PROC / 'fig_10_ekf_soc.html'))
print(f'SOC final CC : {soc_cc[-1]*100:.1f}%  |  EKF : {soc_ekf[-1]*100:.1f}%')


SOC final CC : 8.5%  |  EKF : 7.8%


## 3. Test de robustesse — erreur SOC initiale

In [4]:
# Que se passe-t-il si on démarre avec un SOC initial erroné ?
fig = go.Figure()
colors = ['crimson', 'darkorange', 'seagreen', 'royalblue']

for soc_init, color in zip([0.6, 0.7, 0.8, 1.0], colors):
    ekf.reset(soc0=soc_init)
    soc_test = np.zeros(len(t))
    for k in range(len(t)):
        ekf.predict(I[k], dt_arr[k])
        ekf.update(V[k])
        soc_test[k] = ekf.x[0, 0]
    fig.add_trace(go.Scatter(x=t_min, y=soc_test*100,
        name=f'SOC0 = {soc_init*100:.0f}%', line=dict(color=color, width=1.5)))

fig.add_trace(go.Scatter(x=t_min, y=soc_cc*100, name='CC (ref)',
    line=dict(color='black', dash='dot', width=1)))
fig.update_layout(title='Robustesse EKF — convergence depuis differents SOC initiaux',
    xaxis_title='Temps (min)', yaxis_title='SOC (%)',
    template='plotly_white', height=400)
fig.show()
fig.write_html(str(DATA_PROC / 'fig_11_ekf_robustness.html'))
print('Observation : l EKF converge vers la bonne valeur meme avec un mauvais SOC initial.')


Observation : l EKF converge vers la bonne valeur meme avec un mauvais SOC initial.


---
## Résumé

| Méthode | Avantage | Limite |
|---------|----------|--------|
| Coulomb Counting | Simple | Dérive cumulative |
| EKF | Correction automatique, incertitude quantifiée | Nécessite ECM précis |

**Suivant :** `04_soc_lstm.ipynb` — LSTM pour apprendre SOC directement depuis V, I, T.
